# Week 7 - Evaluation

In [3]:
import os
os.environ["PYTHONUTF8"] = "1"

import re
import sys
sys.path.insert(0, "../src")

import time
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from dotenv import load_dotenv
load_dotenv('../.env')

os.makedirs('../data/eval', exist_ok=True)
os.makedirs('../results', exist_ok=True)

In [4]:
# pymupdf4llm 활용해 만든 VectorDB 호출

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

BASE_DIR = "C:/Users/seohyun/OneDrive/2026/Advanced_RAG"
PDF_PATH = os.path.join(BASE_DIR, 'data', 'registration_of_real_estatee_manual.pdf')
DENSE_DB_PATH = os.path.join(BASE_DIR, 'chroma_db', 'real_estatee_manual')
COLLECTION_NAME = 'real_estatee_manual'

embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

db = Chroma(
    persist_directory=DENSE_DB_PATH,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
)
print(f'기존 ChromaDB 로드: {db._collection.count()}개 문서')

기존 ChromaDB 로드: 327개 문서


In [5]:
# 변경된 임베딩으로 Hybrid Search+Reranking 실험

from typing import List, Any
from sentence_transformers import CrossEncoder
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever


# BM25용 Document 리스트 생성
raw = db.get(include=["documents", "metadatas"])

bm25_docs = [
    Document(page_content=doc, metadata=metadata or {})
    for doc, metadata in zip(raw["documents"], raw["metadatas"])
]


# Cross-Encoder Re-Ranker
def korean_tokenizer(text: str):
    """BM25용 한국어 토크나이저: 특수문자 제거 + 공백 분리 + 1글자 제거"""
    cleaned = re.sub("[^가-힣a-zA-Z0-9]", " ", text)
    return [t for t in cleaned.split() if len(t) > 1]

RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
cross_encoder = CrossEncoder(RERANKER_MODEL)


# Retriever 
bm25_retriever_k20 = BM25Retriever.from_documents(
    bm25_docs,
    k=10,
    preprocess_func=korean_tokenizer,
)

dense_retriever_k20 = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 20},
)

hybrid_retriever_k20 = EnsembleRetriever(
    retrievers=[bm25_retriever_k20, dense_retriever_k20],
    weights=[0.5, 0.5],
    c=60,
)

def hybrid_rerank_retriever(query: str, top_k: int = 5) -> List[Document]:
    """Hybrid 1차 검색 -> Cross-Encoder 재정렬"""
    candidates = hybrid_retriever_k20.invoke(query)

    if not candidates:
        return []

    pairs = [(query, doc.page_content) for doc in candidates]
    scores = cross_encoder.predict(pairs)

    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1354.20it/s]


In [4]:
# 비교 실험

import time
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

from prompt.prompt import GENERATE_PROMPT

with open('../data/eval/testset.json', 'r', encoding='utf-8') as f:
    testset = json.load(f)

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [ ]:
retriever_map = {
    'Dense': lambda q: dense_retriever_k20.invoke(q)[:5],
    'BM25': lambda q: bm25_retriever_k20.invoke(q)[:5],
    'Hybrid': lambda q: hybrid_retriever_k20.invoke(q)[:5],
    'Hybrid+Rerank': lambda q: hybrid_rerank_retriever(q, top_k=5),
}

cmp_results = {name: [] for name in retriever_map}

for name, fn in retriever_map.items():
    print(f'\n[{name}] 추론 시작...')

    for item in testset:
        q = item['question']

        retrieve_start = time.time()
        docs = fn(q)
        retrieve_latency = time.time() - retrieve_start

        context = '\n\n'.join(d.page_content for d in docs)

        generate_start = time.time()
        answer = llm.invoke(
            GENERATE_PROMPT.format_messages(context=context, question=q)
        ).content
        generate_latency = time.time() - generate_start

        cmp_results[name].append({
            'id': item['id'],
            'question': q,
            'contexts': [d.page_content for d in docs],
            'answer': answer,
            'retrieve_latency': retrieve_latency,
            'generate_latency': generate_latency,
            'total_latency': retrieve_latency + generate_latency,
        })

        print(
            f'  Q{item["id"]}: '
            f'retrieve={retrieve_latency:.2f}s, '
            f'generate={generate_latency:.2f}s, '
            f'total={retrieve_latency + generate_latency:.2f}s'
        )


[Dense] 추론 시작...
  Q1: retrieve=4.20s, generate=5.44s, total=9.64s
  Q2: retrieve=0.50s, generate=6.37s, total=6.86s
  Q3: retrieve=0.46s, generate=5.13s, total=5.58s
  Q4: retrieve=0.46s, generate=5.68s, total=6.13s
  Q5: retrieve=1.09s, generate=4.96s, total=6.05s

[BM25] 추론 시작...
  Q1: retrieve=0.01s, generate=4.82s, total=4.83s
  Q2: retrieve=0.01s, generate=4.58s, total=4.59s
  Q3: retrieve=0.01s, generate=4.28s, total=4.29s
  Q4: retrieve=0.00s, generate=7.22s, total=7.22s
  Q5: retrieve=0.00s, generate=4.30s, total=4.31s

[Hybrid] 추론 시작...
  Q1: retrieve=0.44s, generate=4.76s, total=5.20s
  Q2: retrieve=0.66s, generate=3.24s, total=3.90s
  Q3: retrieve=0.67s, generate=7.44s, total=8.11s
  Q4: retrieve=0.51s, generate=4.43s, total=4.94s
  Q5: retrieve=0.38s, generate=4.38s, total=4.76s

[Hybrid+Rerank] 추론 시작...
  Q1: retrieve=550.10s, generate=8.56s, total=558.67s
  Q2: retrieve=499.96s, generate=6.52s, total=506.48s
  Q3: retrieve=393.73s, generate=7.00s, total=400.73s
  Q4: re

In [ ]:
from ragas import evaluate, RunConfig
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithoutReference,
)
from langchain_google_genai import ChatGoogleGenerativeAI

judge_llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0,
    timeout=180,
    max_retries=5,
)

ragas_llm = LangchainLLMWrapper(judge_llm)
ragas_emb = LangchainEmbeddingsWrapper(embeddings)

metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
]

run_config = RunConfig(
    timeout=180,
    max_retries=5,
    max_workers=1,
)

cmp_scores = {}

for name, res_list in cmp_results.items():
    samples = [
        SingleTurnSample(
            user_input=r['question'],
            response=r['answer'],
            retrieved_contexts=r['contexts'],
        )
        for r in res_list
    ]

    print(f'[{name}] RAGAS 평가 중...')

    eval_df = evaluate(
        dataset=EvaluationDataset(samples=samples),
        metrics=metrics,
        llm=ragas_llm,
        embeddings=ragas_emb,
        run_config=run_config,
        raise_exceptions=False,
    ).to_pandas()

    cmp_scores[name] = {
        'Faithfulness': round(float(eval_df['faithfulness'].dropna().mean()), 3),
        'Answer Relevancy': round(float(eval_df['answer_relevancy'].dropna().mean()), 3),
        'Context Precision': round(
            float(eval_df['llm_context_precision_without_reference'].dropna().mean()),
            3
        ),
        'Avg Retrieve Latency(s)': round(
            sum(r['retrieve_latency'] for r in res_list) / len(res_list),
            2
        ),
        'Avg Generate Latency(s)': round(
            sum(r['generate_latency'] for r in res_list) / len(res_list),
            2
        ),
        'Avg Total Latency(s)': round(
            sum(r['total_latency'] for r in res_list) / len(res_list),
            2
        ),
    }

    print(f"  -> {cmp_scores[name]}")

[Dense] RAGAS 평가 중...


Evaluating: 100%|██████████| 15/15 [10:17<00:00, 41.19s/it]


  -> {'Faithfulness': 0.696, 'Answer Relevancy': 0.684, 'Context Precision': 0.773, 'Avg Retrieve Latency(s)': 1.34, 'Avg Generate Latency(s)': 5.52, 'Avg Total Latency(s)': 6.85}
[BM25] RAGAS 평가 중...


Evaluating: 100%|██████████| 15/15 [09:18<00:00, 37.21s/it]


  -> {'Faithfulness': 0.281, 'Answer Relevancy': 0.723, 'Context Precision': 0.536, 'Avg Retrieve Latency(s)': 0.01, 'Avg Generate Latency(s)': 5.04, 'Avg Total Latency(s)': 5.05}
[Hybrid] RAGAS 평가 중...


Evaluating: 100%|██████████| 15/15 [09:44<00:00, 39.00s/it]


  -> {'Faithfulness': 0.83, 'Answer Relevancy': 0.728, 'Context Precision': 0.521, 'Avg Retrieve Latency(s)': 0.53, 'Avg Generate Latency(s)': 4.85, 'Avg Total Latency(s)': 5.38}
[Hybrid+Rerank] RAGAS 평가 중...


Evaluating: 100%|██████████| 15/15 [11:03<00:00, 44.24s/it]

  -> {'Faithfulness': 0.931, 'Answer Relevancy': 0.696, 'Context Precision': 0.646, 'Avg Retrieve Latency(s)': 426.76, 'Avg Generate Latency(s)': 7.36, 'Avg Total Latency(s)': 434.12}


In [ ]:
summary_df = pd.DataFrame(cmp_scores).T
summary_df.index.name = '구성'

print('=' * 70)
print('Dense / BM25 / Hybrid / Hybrid+Rerank — RAGAS 3지표 비교')
print('=' * 70)
display(summary_df)

with open('C:/Users/seohyun/OneDrive/2026/Advanced_RAG/data/result/cmp_results.json', 'w', encoding='utf-8') as f:
    json.dump(cmp_results, f, ensure_ascii=False, indent=2)

with open('C:/Users/seohyun/OneDrive/2026/Advanced_RAG/data/result/cmp_scores.json', 'w', encoding='utf-8') as f:
    json.dump(cmp_scores, f, ensure_ascii=False, indent=2)

# summary_df.to_csv('../data/eval/retriever_comparison.csv', encoding='utf-8-sig')
# print('\nretriever_comparison.csv 저장 완료')

Dense / BM25 / Hybrid / Hybrid+Rerank — RAGAS 3지표 비교


,Faithfulness,Answer Relevancy,Context Precision,Avg Retrieve Latency(s),Avg Generate Latency(s),Avg Total Latency(s)
구성,,,,,,
Dense,0.696,0.684,0.773,1.34,5.52,6.85
BM25,0.281,0.723,0.536,0.01,5.04,5.05
Hybrid,0.830,0.728,0.521,0.53,4.85,5.38
Hybrid+Rerank,0.931,0.696,0.646,426.76,7.36,434.12


### Agentic RAG (LangGraph)

In [5]:
from typing import Any
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from prompt.prompt import GRADE_PROMPT, REWRITE_PROMPT, GENERATE_PROMPT

class GraphState(TypedDict):
    question: str
    rewritten_question: str
    documents: List[Any]
    answer: str
    grade_result: str
    retry_count: int
    route_history: list
    latency: float

MAX_RETRIES = 2

class GradeResult(BaseModel):
    relevance: str = Field(description="'yes'/'no'")
    reason: str = Field(description='판단 이유')

grade_llm = llm.with_structured_output(GradeResult)

def retrieve(state: GraphState) -> dict:
    q = state.get('rewritten_question') or state['question']
    docs = hybrid_rerank_retriever(q, top_k=5)
    history = list(state.get('route_history') or [])
    history.append('retrieve')
    return {'documents': docs, 'grade_result': '', 'answer': '', 'route_history': history}

def grade_documents(state: GraphState) -> dict:
    q = state.get('rewritten_question') or state['question']
    docs = state['documents']
    if not docs:
        history = list(state.get('route_history') or [])
        history.append('grade=no(empty)')
        return {'grade_result': 'no', 'route_history': history}
    doc_previews = '\n\n'.join([
        f'[문서 {i+1}] {doc.page_content[:250]}'
        for i, doc in enumerate(docs[:5])
    ])
    result = grade_llm.invoke(GRADE_PROMPT.format_messages(question=q, doc_previews=doc_previews))
    history = list(state.get('route_history') or [])
    history.append(f'grade={result.relevance}')
    return {'grade_result': result.relevance, 'route_history': history}

def rewrite_query(state: GraphState) -> dict:
    current_q = state.get('rewritten_question') or state['question']
    retry_count = state.get('retry_count') or 0
    response = llm.invoke(REWRITE_PROMPT.format_messages(question=current_q))
    rewritten = response.content.strip()
    new_retry = retry_count + 1
    history = list(state.get('route_history') or [])
    history.append(f'rewrite({new_retry})')
    return {'rewritten_question': rewritten, 'retry_count': new_retry, 'route_history': history}

def generate(state: GraphState) -> dict:
    q = state.get('rewritten_question') or state['question']
    docs = state.get('documents') or []
    grade_result = state.get('grade_result', 'no')
    history = list(state.get('route_history') or [])
    if grade_result != 'yes' or not docs:
        history.append('generate(refusal)')
        return {
            'answer': '제공된 문서에서 확인할 수 없습니다. 관련 전문가와 상담하시기 바랍니다.',
            'route_history': history,
        }
    context = '\n\n'.join([doc.page_content for doc in docs])
    response = llm.invoke(GENERATE_PROMPT.format_messages(context=context, question=q))
    history.append('generate(success)')
    return {'answer': response.content, 'route_history': history}

def route_after_grade(state: GraphState) -> str:
    grade = state.get('grade_result', 'no')
    retry = state.get('retry_count') or 0
    if grade == 'yes':
        return 'generate'
    elif retry >= MAX_RETRIES:
        return 'generate'
    else:
        return 'rewrite_query'

workflow = StateGraph(GraphState)
workflow.add_node('retrieve', retrieve)
workflow.add_node('grade_documents', grade_documents)
workflow.add_node('rewrite_query', rewrite_query)
workflow.add_node('generate', generate)
workflow.set_entry_point('retrieve')
workflow.add_edge('retrieve', 'grade_documents')
workflow.add_conditional_edges(
    'grade_documents',
    route_after_grade,
    {'generate': 'generate', 'rewrite_query': 'rewrite_query'},
)
workflow.add_edge('rewrite_query', 'retrieve')
workflow.add_edge('generate', END)
agentic_app = workflow.compile()

print('Agentic RAG (LangGraph) 컴파일 완료')
print(agentic_app.get_graph().draw_mermaid())

Agentic RAG (LangGraph) 컴파일 완료
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade_documents(grade_documents)
	rewrite_query(rewrite_query)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	grade_documents -.-> generate;
	grade_documents -.-> rewrite_query;
	retrieve --> grade_documents;
	rewrite_query --> retrieve;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [53]:
def run_baseline(query: str) -> dict:
    """Baseline: Hybrid Search -> 직접 생성"""
    retrieve_start = time.time()
    docs = hybrid_retriever_k20.invoke(query)[:5]
    retrieve_latency = time.time() - retrieve_start

    context = '\n\n'.join(doc.page_content for doc in docs)

    generate_start = time.time()
    response = llm.invoke(
        GENERATE_PROMPT.format_messages(context=context, question=query)
    )
    generate_latency = time.time() - generate_start

    return {
        'answer': response.content,
        'contexts': [
            {'content': doc.page_content, 'section_title': doc.metadata.get('section_title', '')}
            for doc in docs
        ],
        'route_history': ['retrieve', 'generate(direct)'],
        'retrieve_latency': retrieve_latency,
        'generate_latency': generate_latency,
        'total_latency': retrieve_latency + generate_latency,
        'latency': retrieve_latency + generate_latency,
    }


def run_agentic(query: str) -> dict:
    """Agentic RAG: LangGraph 실행"""
    start = time.time()

    initial_state: GraphState = {
        'question': query,
        'rewritten_question': '',
        'documents': [],
        'answer': '',
        'grade_result': '',
        'retry_count': 0,
        'route_history': [],
        'latency': 0.0,
    }

    final_state = agentic_app.invoke(initial_state)
    total_latency = time.time() - start

    final_docs = final_state.get('documents') or []

    return {
        'answer': final_state['answer'],
        'contexts': [
            {'content': doc.page_content, 'section_title': doc.metadata.get('section_title', '')}
            for doc in final_docs
        ],
        'grade_result': final_state.get('grade_result', ''),
        'retry_count': final_state.get('retry_count', 0),
        'rewritten_question': final_state.get('rewritten_question', ''),
        'route_history': final_state.get('route_history', []),
        'retrieve_latency': None,
        'generate_latency': None,
        'total_latency': total_latency,
        'latency': total_latency,
    }

In [ ]:
# Agentic RAG test query

cmp_results['Agentic'] = []

for item in testset:
    q = item['question']

    print(f'Q{item["id"]} Agentic 실행 중...')
    agentic_result = run_agentic(q)
    cmp_results['Agentic'].append({
        'id': item['id'],
        'question': q,
        **agentic_result,
    })

    print(
        f'Agentic={agentic_result["total_latency"]:.2f}s'
    )

Q1 Agentic 실행 중...
Agentic=534.71s
Q2 Agentic 실행 중...
Agentic=358.06s
Q3 Agentic 실행 중...
Agentic=488.30s
Q4 Agentic 실행 중...
Agentic=508.03s
Q5 Agentic 실행 중...
Agentic=321.26s


In [ ]:
name = 'Agentic'
res_list = cmp_results[name]

samples = [
    SingleTurnSample(
        user_input=r['question'],
        response=r['answer'],
        retrieved_contexts=[c['content'] for c in r['contexts']],
    )
    for r in res_list
]

print(f'[{name}] RAGAS 평가 중...')

eval_df = evaluate(
    dataset=EvaluationDataset(samples=samples),
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=run_config,
    raise_exceptions=False,
).to_pandas()

cmp_scores[name] = {
    'Faithfulness': round(float(eval_df['faithfulness'].mean(skipna=True)), 3),
    'Answer Relevancy': round(float(eval_df['answer_relevancy'].mean(skipna=True)), 3),
    'Context Precision': round(
        float(eval_df['llm_context_precision_without_reference'].mean(skipna=True)),
        3
    ),
    'Avg Total Latency(s)': round(
        sum(r['total_latency'] for r in res_list) / len(res_list),
        2
    ),
    'Avg Retry Count': round(
        sum(r.get('retry_count', 0) for r in res_list) / len(res_list),
        2
    ),
}

print(f"  -> {cmp_scores[name]}")

### Golden set 테스트

In [54]:
# Golden Set

with open('../data/eval/golden_set_v1.json', 'r', encoding='utf-8') as f:
    golden_set = json.load(f)

In [55]:
golden_df = pd.DataFrame(golden_set)
print(f'Golden Set: 총 {len(golden_df)}개 문항')
print(golden_df['q_type'].value_counts().to_string())

Golden Set: 총 20개 문항
q_type
factual         4
comparison      4
procedural      4
out_of_scope    4
safety          4


In [56]:
baseline_all = []
agentic_all  = []

for i, row in golden_df.iterrows():
    q = row['question']
    print(f'\nQ{i+1:02d} [{row["q_type"]:12s}] {q[:45]}')

    b_res = run_baseline(q)
    baseline_all.append(b_res)

    a_res = run_agentic(q)
    agentic_all.append(a_res)

    print(f'  Baseline : {b_res["total_latency"]:.1f}s')
    print(f'  Agentic  : {a_res["total_latency"]:.1f}s  retry={a_res["retry_count"]}  grade={a_res["grade_result"]}')


Q01 [factual     ] 대출 다 갚고 나서 근저당 해제할 때, 신청하는 사람 두 명 중에 권리자가 돈 빌
  Baseline : 10.4s
  Agentic  : 638.7s  retry=1  grade=yes

Q02 [factual     ] 집 새로 지어서 처음으로 내 집이라고 등기 올릴 때 소유자라는 거 입증하는 서류가
  Baseline : 6.1s
  Agentic  : 205.5s  retry=0  grade=yes

Q03 [factual     ] 소유권이전등기 신청서 서식에서 '등기의무자' 칸에는 매도인과 매수인 중 누구 정보
  Baseline : 3.2s
  Agentic  : 190.0s  retry=0  grade=yes

Q04 [factual     ] 미성년자 명의로 상속 부동산을 취득한 경우, 등기 신청인과 필요 서류를 알려주세요
  Baseline : 5.2s
  Agentic  : 198.8s  retry=0  grade=yes

Q05 [comparison  ] 돌아가신 분 재산 물려받는 경우랑, 살아 계실 때 집을 그냥 줄 때 등기 신청하는
  Baseline : 1.8s
  Agentic  : 334.2s  retry=0  grade=yes

Q06 [comparison  ] 계약금만 내고 일단 걸어두는 가계약 등기랑, 잔금까지 다 치른 뒤 완전히 소유권 
  Baseline : 5.7s
  Agentic  : 683.1s  retry=2  grade=no

Q07 [comparison  ] 집에 담보 잡을 때랑 담보 풀 때 등기 신청인이 서로 바뀌나요?
  Baseline : 3.2s
  Agentic  : 501.6s  retry=2  grade=yes

Q08 [comparison  ] 공유 토지를 협의로 나눌 때랑 법원 판결로 강제로 나눌 때, 등기 신청하는 사람 
  Baseline : 5.9s
  Agentic  : 375.3s  retry=1  grade=yes

Q09 [procedural  

In [57]:
with open('C:/Users/seohyun/OneDrive/2026/Advanced_RAG/data/result/baseline_all.json', 'w', encoding='utf-8') as f:
    json.dump(baseline_all, f, ensure_ascii=False, indent=2)

with open('C:/Users/seohyun/OneDrive/2026/Advanced_RAG/data/result/agentic_all.json', 'w', encoding='utf-8') as f:
    json.dump(agentic_all, f, ensure_ascii=False, indent=2)

---

In [6]:
import json
import pandas as pd

RESULT_DIR = 'C:/Users/seohyun/OneDrive/2026/Advanced_RAG/data/result'

with open(f'{RESULT_DIR}/baseline_all.json', 'r', encoding='utf-8') as f:
    baseline_all = json.load(f)

with open(f'{RESULT_DIR}/agentic_all.json', 'r', encoding='utf-8') as f:
    agentic_all = json.load(f)

with open('../data/eval/golden_set_v1.json', 'r', encoding='utf-8') as f:
    golden_set = json.load(f)

golden_df = pd.DataFrame(golden_set)

REFUSAL_KEYWORDS = [
    '확인할 수 없', '제공된 문서에서', '전문가와 상담', '불법', '범죄',
    '세무사', '법률 전문가', '기능 범위', '신고하시기',
]

def is_refusal(answer: str) -> bool:
    return any(kw in answer for kw in REFUSAL_KEYWORDS)

print(f'baseline_all: {len(baseline_all)}개')
print(f'agentic_all : {len(agentic_all)}개')
print(f'golden_set  : {len(golden_df)}개')

baseline_all: 20개
agentic_all : 20개
golden_set  : 20개


In [8]:
# RAGAS 4지표 — Baseline vs Agentic (factual/comparison/procedural만)
from ragas import evaluate, RunConfig
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, AnswerRelevancy, LLMContextPrecisionWithoutReference, LLMContextRecall
from langchain_google_genai import ChatGoogleGenerativeAI

judge_llm  = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0, timeout=180, max_retries=5)
ragas_llm  = LangchainLLMWrapper(judge_llm)
ragas_emb  = LangchainEmbeddingsWrapper(embeddings)
run_config = RunConfig(timeout=180, max_retries=5, max_workers=1)

metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
    LLMContextRecall(llm=ragas_llm),
]

EVAL_TYPES = {'factual', 'comparison', 'procedural'}  # out_of_scope/safety 제외

def make_samples(result_list):
    return [
        SingleTurnSample(
            user_input=row['question'],
            response=r['answer'],
            retrieved_contexts=[c['content'] for c in r['contexts']] or ['검색 결과 없음'],
            reference=row['ground_truth'],
        )
        for (_, row), r in zip(golden_df.iterrows(), result_list)
        if row['q_type'] in EVAL_TYPES
    ]

print('Baseline RAGAS 평가 중…')
baseline_eval_df = evaluate(
    dataset=EvaluationDataset(samples=make_samples(baseline_all)),
    metrics=metrics, llm=ragas_llm, embeddings=ragas_emb,
    run_config=run_config, raise_exceptions=False,
).to_pandas()
print('완료')

Baseline RAGAS 평가 중…


Evaluating: 100%|██████████| 48/48 [16:01<00:00, 20.04s/it]

완료


In [10]:
print('Agentic RAGAS 평가 중…')
agentic_eval_df = evaluate(
    dataset=EvaluationDataset(samples=make_samples(agentic_all)),
    metrics=metrics, llm=ragas_llm, embeddings=ragas_emb,
    run_config=run_config, raise_exceptions=False,
).to_pandas()
print('완료')

Agentic RAGAS 평가 중…


Evaluating: 100%|██████████| 48/48 [30:47<00:00, 38.49s/it] 

완료


In [11]:
RAGAS_COLS = ['faithfulness', 'answer_relevancy', 'llm_context_precision_without_reference', 'context_recall']
LABEL_MAP  = {
    'faithfulness': 'Faithfulness',
    'answer_relevancy': 'AnswerRel',
    'llm_context_precision_without_reference': 'CtxPrecision',
    'context_recall': 'CtxRecall',
}

b_means = baseline_eval_df[RAGAS_COLS].mean(skipna=True).rename(LABEL_MAP)
a_means = agentic_eval_df[RAGAS_COLS].mean(skipna=True).rename(LABEL_MAP)

cmp = pd.DataFrame({'Baseline': b_means, 'Agentic': a_means})
cmp['Delta(A-B)'] = (cmp['Agentic'] - cmp['Baseline']).round(3)
print('=== RAGAS 4지표 (전체 평균) ===')
print(cmp.round(3).to_string())

lats_b = pd.Series([r['total_latency'] for r in baseline_all])
lats_a = pd.Series([r['total_latency'] for r in agentic_all])
print(f'\nLatency (s)  Baseline mean={lats_b.mean():.2f}  Agentic mean={lats_a.mean():.2f}  Delta={lats_a.mean()-lats_b.mean():+.2f}')

=== RAGAS 4지표 (전체 평균) ===
              Baseline  Agentic  Delta(A-B)
Faithfulness     0.476    0.662       0.186
AnswerRel        0.692    0.654      -0.037
CtxPrecision     0.547    0.828       0.281
CtxRecall        0.194    0.417       0.222

Latency (s)  Baseline mean=4.80  Agentic mean=316.14  Delta=+311.34


In [12]:
# ── 1. Latency & Retry 통계 ──────────────────────────────────────────
rows = []
for i, row in golden_df.iterrows():
    b = baseline_all[i]
    a = agentic_all[i]
    rows.append({
        'q_type'         : row['q_type'],
        'b_latency'      : b['total_latency'],
        'a_latency'      : a['total_latency'],
        'overhead'       : a['total_latency'] - b['total_latency'],
        'retry_count'    : a.get('retry_count', 0),
        'grade_result'   : a.get('grade_result', ''),
        'a_is_refusal'   : is_refusal(a['answer']),
        'b_is_refusal'   : is_refusal(b['answer']),
    })

stats_df = pd.DataFrame(rows)

print('[ 전체 평균 ]')
print(f"  Baseline 평균 latency : {stats_df['b_latency'].mean():.1f}s")
print(f"  Agentic  평균 latency : {stats_df['a_latency'].mean():.1f}s")
print(f"  평균 overhead         : +{stats_df['overhead'].mean():.1f}s")
print(f"  평균 retry 횟수       : {stats_df['retry_count'].mean():.2f}")
print()

print('[ q_type별 평균 latency & retry ]')
type_stats = stats_df.groupby('q_type').agg(
    문항수=('b_latency', 'count'),
    Baseline_lat=('b_latency', 'mean'),
    Agentic_lat=('a_latency', 'mean'),
    Overhead=('overhead', 'mean'),
    Avg_retry=('retry_count', 'mean'),
).round(1)
display(type_stats)

[ 전체 평균 ]
  Baseline 평균 latency : 4.8s
  Agentic  평균 latency : 316.1s
  평균 overhead         : +311.3s
  평균 retry 횟수       : 0.85

[ q_type별 평균 latency & retry ]


,문항수,Baseline_lat,Agentic_lat,Overhead,Avg_retry
q_type,,,,,
comparison,4,4.2,473.6,469.4,1.2
factual,4,6.2,308.2,302.0,0.2
out_of_scope,4,2.0,311.9,309.8,1.2
procedural,4,7.4,189.5,182.1,0.0
safety,4,4.2,297.5,293.4,1.5


In [13]:
# ── 2. Refusal Accuracy ───────────────────────────────────────────────
REFUSAL_TYPES = {'out_of_scope', 'safety'}
ANSWER_TYPES  = {'factual', 'comparison', 'procedural', 'multi_hop'}

refusal_idx = golden_df[golden_df['q_type'].isin(REFUSAL_TYPES)].index.tolist()
answer_idx  = golden_df[golden_df['q_type'].isin(ANSWER_TYPES)].index.tolist()

for label, results in [('Baseline', baseline_all), ('Agentic', agentic_all)]:
    correct   = sum(1 for i in refusal_idx if is_refusal(results[i]['answer']))
    false_ref = sum(1 for i in answer_idx  if is_refusal(results[i]['answer']))
    print(f'[{label}]  거절 정확도: {correct}/{len(refusal_idx)} = {correct/len(refusal_idx):.2f}'
          f'   오거절: {false_ref}건')

print()
print('[ q_type별 거절/답변 현황 ]')
for q_type in sorted(golden_df['q_type'].unique()):
    idx_list = golden_df[golden_df['q_type'] == q_type].index.tolist()
    b_ref = sum(1 for i in idx_list if is_refusal(baseline_all[i]['answer']))
    a_ref = sum(1 for i in idx_list if is_refusal(agentic_all[i]['answer']))
    tag   = '← 거절해야 함' if q_type in REFUSAL_TYPES else ''
    print(f'  {q_type:15s} ({len(idx_list)}개)  Baseline 거절={b_ref}  Agentic 거절={a_ref}  {tag}')

[Baseline]  거절 정확도: 0/8 = 0.00   오거절: 1건
[Agentic]  거절 정확도: 5/8 = 0.62   오거절: 1건

[ q_type별 거절/답변 현황 ]
  comparison      (4개)  Baseline 거절=0  Agentic 거절=1  
  factual         (4개)  Baseline 거절=0  Agentic 거절=0  
  out_of_scope    (4개)  Baseline 거절=0  Agentic 거절=2  ← 거절해야 함
  procedural      (4개)  Baseline 거절=1  Agentic 거절=0  
  safety          (4개)  Baseline 거절=0  Agentic 거절=3  ← 거절해야 함


In [14]:
# ── 3. Q별 비교 테이블 ────────────────────────────────────────────────
compare_rows = []
for i, row in golden_df.iterrows():
    b = baseline_all[i]
    a = agentic_all[i]
    compare_rows.append({
        'No'           : i + 1,
        'q_type'       : row['q_type'],
        'question'     : row['question'][:40] + '...',
        'b_answer'     : b['answer'][:80] + '...' if len(b['answer']) > 80 else b['answer'],
        'a_answer'     : a['answer'][:80] + '...' if len(a['answer']) > 80 else a['answer'],
        'retry'        : a.get('retry_count', 0),
        'grade'        : a.get('grade_result', ''),
        'route'        : ' → '.join(a.get('route_history', [])),
        'b_lat(s)'     : round(b['total_latency'], 1),
        'a_lat(s)'     : round(a['total_latency'], 1),
    })

compare_df = pd.DataFrame(compare_rows).set_index('No')
pd.set_option('display.max_colwidth', 90)
display(compare_df[['q_type', 'question', 'b_answer', 'a_answer', 'retry', 'grade', 'b_lat(s)', 'a_lat(s)']])

,q_type,question,b_answer,a_answer,retry,grade,b_lat(s),a_lat(s)
No,,,,,,,,
1,factual,"대출 다 갚고 나서 근저당 해제할 때, 신청하는 사람 두 명 중에 권리자...","근저당 해제를 신청하는 경우, 신청하는 사람은 근저당권의 권리자, 즉 돈을 빌려준 사람입니다. 근저당권은 채권자가 채무자의 채무를 담보하기 위해...","대출 상환 후 근저당 해제 등기 신청 시, 권리자와 채무자 간의 관계는 다음과 같습니다:\n\n- **등기의무자**: 근저당권 설정자 (채무자)\n-...",1,yes,10.4,638.7
2,factual,집 새로 지어서 처음으로 내 집이라고 등기 올릴 때 소유자라는 거 입증하...,집을 새로 지어서 처음으로 소유권 등기를 올릴 때 필요한 서류는 다음과 같습니다:\n\n1. **건축물대장등본**: 해당 건축물의 등록 사항을 확인...,집을 새로 지어서 처음으로 등기를 올릴 때 소유자임을 입증하는 서류는 다음과 같습니다:\n\n1. **소유권을 증명하는 서면**: 일반적으로는 매매...,0,yes,6.1,205.5
3,factual,소유권이전등기 신청서 서식에서 '등기의무자' 칸에는 매도인과 매수인 중 ...,"소유권이전등기 신청서 서식에서 '등기의무자' 칸에는 매도인의 정보를 적어야 합니다. 등기의무자는 매도인으로, 등기권리자는 매수인입니다. 이는 「...","소유권이전등기 신청서 서식에서 '등기의무자' 칸에는 매도인의 정보를 적어야 합니다. 등기의무자는 매도인으로, 등기권리자는 매수인으로 기재됩니다....",0,yes,3.2,190.0
4,factual,"미성년자 명의로 상속 부동산을 취득한 경우, 등기 신청인과 필요 서류를 ...","미성년자 명의로 상속 부동산을 취득한 경우, 등기 신청인은 미성년자의 법정대리인(부모 또는 후견인)입니다. \n\n필요 서류는 다음과 같습니다:\n\n...","미성년자 명의로 상속 부동산을 취득한 경우, 등기 신청인은 미성년자의 법정대리인(부모 또는 후견인)이 됩니다. \n\n필요 서류는 다음과 같습니다:...",0,yes,5.2,198.8
5,comparison,"돌아가신 분 재산 물려받는 경우랑, 살아 계실 때 집을 그냥 줄 때 등기...",확인 불가,"네, 돌아가신 분의 재산을 물려받는 경우와 살아 계실 때 집을 주는 경우 등기 신청하는 사람의 구성이 다릅니다.\n\n1. **상속의 경우**: 돌...",0,yes,1.8,334.2
6,comparison,"계약금만 내고 일단 걸어두는 가계약 등기랑, 잔금까지 다 치른 뒤 완전히...",가계약 등기와 완전한 소유권 이전 등기는 법적으로 다음과 같은 차이가 있습니다.\n\n1. **가계약 등기**: 가계약은 계약금만 지급하고 나머지 ...,제공된 문서에서 확인할 수 없습니다. 관련 전문가와 상담하시기 바랍니다.,2,no,5.7,683.1
7,comparison,집에 담보 잡을 때랑 담보 풀 때 등기 신청인이 서로 바뀌나요?...,담보를 잡을 때와 담보를 풀 때 등기 신청인이 서로 바뀌는 경우가 있습니다. \n\n1. **담보 잡을 때**: 일반적으로 담보를 설정할 때는 채무...,"담보권 설정 등기 신청인과 담보권 말소 등기 신청인이 변경되는 경우, 법적 절차 및 요건은 다음과 같습니다.\n\n1. **담보권 이전**: 담보권...",2,yes,3.2,501.6
8,comparison,"공유 토지를 협의로 나눌 때랑 법원 판결로 강제로 나눌 때, 등기 신청하...",공유 토지를 협의로 나눌 때와 법원 판결로 강제로 나눌 때의 등기 신청하는 사람 구성은 다릅니다.\n\n1. **협의에 의한 분할**: 공유자들이 ...,공유 토지를 협의에 의한 분할 등기 신청 시와 법원 판결에 의한 강제 분할 등기 신청 시의 등기 신청인 구성 및 필요한 서류 절차는 다음과 같이...,1,yes,5.9,375.3
9,procedural,아버지가 갑자기 돌아가셨어요. 아버지 명의 아파트를 제 이름으로 바꾸려면...,아버지 명의의 아파트를 본인 명의로 이전하기 위해서는 상속등기를 진행해야 합니다. 다음은 그 절차입니다:\n\n1. **상속인 확인**: 아버지의 ...,"아버지의 명의로 되어 있는 아파트를 당신의 이름으로 바꾸기 위해서는 상속 절차를 거쳐야 합니다. 아버지가 돌아가신 후, 상속인인 당신과 다른 가...",0,yes,7.3,235.4


In [15]:
# ── 4. Routing 상세 분석 ──────────────────────────────────────────────
print('[ Agentic Routing 상세 ]')
print(f"{'No':>3}  {'q_type':12}  {'retry':>5}  {'grade':>5}  route")
print('-' * 75)
for i, row in golden_df.iterrows():
    a = agentic_all[i]
    route_str = ' → '.join(a.get('route_history', []))
    print(f"Q{i+1:02d}  {row['q_type']:12}  {a.get('retry_count', 0):>5}  "
          f"{a.get('grade_result', ''):>5}  {route_str}")

print()
# retry가 발생한 케이스에서 rewrite된 질문 확인
print('[ Query Rewrite 발생 케이스 ]')
for i, row in golden_df.iterrows():
    a = agentic_all[i]
    if a.get('retry_count', 0) > 0 and a.get('rewritten_question', ''):
        print(f"\nQ{i+1} [{row['q_type']}]")
        print(f"  원본    : {row['question']}")
        print(f"  재작성  : {a['rewritten_question']}")
        print(f"  결과    : grade={a.get('grade_result', '')}  → {a['answer'][:60]}...")

[ Agentic Routing 상세 ]
 No  q_type        retry  grade  route
---------------------------------------------------------------------------
Q01  factual           1    yes  retrieve → grade=no → rewrite(1) → retrieve → grade=yes → generate(success)
Q02  factual           0    yes  retrieve → grade=yes → generate(success)
Q03  factual           0    yes  retrieve → grade=yes → generate(success)
Q04  factual           0    yes  retrieve → grade=yes → generate(success)
Q05  comparison        0    yes  retrieve → grade=yes → generate(success)
Q06  comparison        2     no  retrieve → grade=no → rewrite(1) → retrieve → grade=no → rewrite(2) → retrieve → grade=no → generate(refusal)
Q07  comparison        2    yes  retrieve → grade=no → rewrite(1) → retrieve → grade=no → rewrite(2) → retrieve → grade=yes → generate(success)
Q08  comparison        1    yes  retrieve → grade=no → rewrite(1) → retrieve → grade=yes → generate(success)
Q09  procedural        0    yes  retrieve → grade=yes → gener